# Quantitative Study 01: Derivatives Microstructure & OI Alphas
**Author**: Quantitative Research Desk  
**Objective**: Evaluate predictive power of intraday Open Interest (OI) changes, analyze retail "long-buildup" flow, and validate the Inverted OI Fader alpha on NSE weekly options.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from options_bt.chain_parser import OptionsChainParser
from options_bt.oi_fader import OIFaderEngine
from options_bt.strike_matrix import StrikeMatrixEngine

### 1. Ingest Options Chain & Resample to 5-Minute Bars

In [ ]:
options_path = "../sample_data/options/NIFTY_20250101.csv"
parser = OptionsChainParser(options_path)
call_strikes, put_strikes = parser.extract_available_strikes()
print(f"Discovered {len(call_strikes)} Call strikes and {len(put_strikes)} Put strikes.")

atm_strike = 24150
contract_df = parser.get_contract_series(atm_strike, "CE")
contract_5m = parser.resample_bars(contract_df, "5min")
print(f"Resampled {len(contract_5m)} 5-minute candles for {atm_strike}CE.")
contract_5m.head()

### 2. Evaluate Inverted Long-Buildup Fade Signals

In [ ]:
engine = OIFaderEngine(min_price_change_pct=0.03, min_oi_change_pct=0.05)
signals = engine.generate_signals(contract_5m, atm_strike, "CE")
print(f"Generated {len(signals)} institutional fade signals for {atm_strike}CE.")

sig_df = pd.DataFrame([vars(s) for s in signals])
sig_df

### 3. Visualizing Price vs. Open Interest Dynamics

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(contract_5m["DateTime"], contract_5m["price"], color="royalblue", lw=1.8, label="24150 CE Price")
ax2.plot(contract_5m["DateTime"], contract_5m["oi"], color="darkorange", lw=1.5, ls="--", label="24150 CE Open Interest")

ax1.set_xlabel("Time")
ax1.set_ylabel("Option Premium (₹)", color="royalblue")
ax2.set_ylabel("Open Interest (Contracts)", color="darkorange")
plt.title(f"NIFTY {atm_strike}CE: Intraday Price vs Open Interest Trajectory")
plt.show()